[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/ai-ml-interviews-study-guide/blob/main/notebooks/eval_design_case_study.ipynb)

# Designing an Eval End-to-End: A METR-Style Case Study

We are comparing a new agent scaffold, *ReAct++*, against a *plain* baseline on a synthetic agentic-coding task suite, using METR's time-horizon framing (the task length at which the agent solves 50% of tasks). This notebook walks through the full design pipeline: pre-registration, sample-size analysis, synthetic data generation, logistic fitting, hierarchical bootstrap, decision, and sensitivity checks.

By the end you will have produced:
- a pre-registered hypothesis, alpha, power target, and practical threshold;
- a synthetic paired eval with explicit task-level jitter;
- MLE fits of a two-parameter logistic to each scaffold, giving 50%-horizon point estimates;
- a three-level hierarchical bootstrap CI on the log-ratio of horizons;
- a sample-size vs power curve and a minimum `tasks_per_bucket` for 80% power;
- a sensitivity check against a coarser bucket scheme;
- a 90-second interview pitch filled in with the notebook's actual numbers.

## 1. Pre-registration

We write this **before looking at any data**. Pre-registration matters because once you have seen the results, every design choice (how to bin, how to bootstrap, when to stop collecting data) becomes entangled with the outcome you want; pre-committing removes that degree of freedom and makes the eval genuinely confirmatory instead of exploratory.

**Estimand.** Let $H_{\text{plain}}$ and $H_{\text{plus}}$ be the 50%-success time horizons (in minutes) for the two scaffolds, defined as the task length at which the fitted logistic crosses 0.5. Our target of inference is the log-ratio $\Delta = \log(H_{\text{plus}} / H_{\text{plain}})$.

**Hypotheses.**

$$H_0: \Delta \leq 0 \qquad H_1: \Delta > 0$$

**Alpha.** One-sided $\alpha = 0.05$.

**Power target.** $1 - \beta = 0.80$ against the alternative $\Delta = \log(2)$ (a 2x horizon improvement).

**Practical significance threshold.** $\Delta^\star = \log(1.5)$. A statistically significant but sub-50% horizon gain is not an interesting product result.

**Decision rule.** Fit a two-parameter logistic (per scaffold) by MLE, bootstrap the log-ratio with a three-level hierarchical resample (buckets -> tasks -> attempts), take the 95% percentile CI. Declare:
- **Statistically significant** if the one-sided lower bound of $\Delta$ at $\alpha = 0.05$ is $> 0$.
- **Practically significant** if that same lower bound is $> \log(1.5)$.

**Stopping rule.** We pre-committed to the full task set (10 buckets x 20 tasks x 3 attempts per scaffold) before running any analysis. No peeking, no early stopping.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm

np.random.seed(2024)

## 2. Pre-specified parameters

In [ ]:
seed = 2024
length_buckets_minutes = [1, 2, 5, 15, 30, 60, 120, 240, 480, 960]
tasks_per_bucket = 20
attempts_per_task = 3
alpha = 0.05
power_target = 0.80
practical_threshold_log_ratio = np.log(1.5)
B_bootstrap = 2000

# Latent data-generating model
mu_plain = np.log(30.0)   # true 50% horizon = 30 min
mu_plus  = np.log(60.0)   # true 50% horizon = 60 min
beta = 1.2                # shared slope

print(f"True horizon plain: {np.exp(mu_plain):.1f} min")
print(f"True horizon plus:  {np.exp(mu_plus):.1f} min")
print(f"True log-ratio:     {mu_plus - mu_plain:.4f}  (log(2) = {np.log(2):.4f})")
print(f"Practical threshold log_ratio: {practical_threshold_log_ratio:.4f}  (log(1.5))")

## 3. Synthetic data generator

For each (bucket, task) pair we draw a task-level difficulty jitter $\epsilon_j \sim \mathcal{N}(0, 0.15)$. Tasks within the same bucket therefore vary in difficulty, which matters because the bootstrap's middle level (tasks) has to have real variance for its CI to be honest. We pair the two scaffolds on the same $\epsilon_j$ values, which models the realistic case where you run both agents on the same task set.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def generate_eval_data(mu, beta, length_buckets, tasks_per_bucket, attempts_per_task, task_epsilons, rng):
    """Generate a flat attempts array for a single scaffold.

    Returns a structured array with (bucket_idx, task_id, length_minutes, attempt_idx, success).
    task_epsilons is pre-drawn shape (n_buckets, tasks_per_bucket) so the same
    difficulty jitters are shared across scaffolds (paired design).
    """
    n_buckets = len(length_buckets)
    N = n_buckets * tasks_per_bucket * attempts_per_task
    bucket_idx = np.repeat(np.arange(n_buckets), tasks_per_bucket * attempts_per_task)
    task_id = np.tile(np.repeat(np.arange(tasks_per_bucket), attempts_per_task), n_buckets)
    attempt_idx = np.tile(np.arange(attempts_per_task), n_buckets * tasks_per_bucket)
    length_minutes = np.array(length_buckets, dtype=np.float64)[bucket_idx]

    log_len = np.log(length_minutes)
    eps_flat = task_epsilons[bucket_idx, task_id]
    p = sigmoid((mu + eps_flat - log_len) * beta)
    success = (rng.random(N) < p).astype(np.int32)

    dtype = [
        ("bucket_idx", np.int32),
        ("task_id", np.int32),
        ("length_minutes", np.float64),
        ("attempt_idx", np.int32),
        ("success", np.int32),
    ]
    out = np.empty(N, dtype=dtype)
    out["bucket_idx"] = bucket_idx
    out["task_id"] = task_id
    out["length_minutes"] = length_minutes
    out["attempt_idx"] = attempt_idx
    out["success"] = success
    return out


n_buckets = len(length_buckets_minutes)
rng_eps = np.random.default_rng(seed)
task_epsilons = rng_eps.normal(0.0, 0.15, size=(n_buckets, tasks_per_bucket))

rng_plain = np.random.default_rng(seed + 1)
rng_plus  = np.random.default_rng(seed + 2)
data_plain = generate_eval_data(mu_plain, beta, length_buckets_minutes, tasks_per_bucket, attempts_per_task, task_epsilons, rng_plain)
data_plus  = generate_eval_data(mu_plus,  beta, length_buckets_minutes, tasks_per_bucket, attempts_per_task, task_epsilons, rng_plus)

print(f"n attempts per scaffold: {len(data_plain)}")
print(f"overall success rate plain: {data_plain['success'].mean():.3f}")
print(f"overall success rate plus:  {data_plus['success'].mean():.3f}")

## 4. Raw-data visualization

Per-bucket empirical success rates on a log-x axis. The vertical lines mark the true 50% horizons the data was generated from.

In [ ]:
def bucket_success_rates(data, n_buckets):
    rates = np.zeros(n_buckets)
    lengths = np.zeros(n_buckets)
    for i in range(n_buckets):
        mask = data["bucket_idx"] == i
        rates[i] = data["success"][mask].mean()
        lengths[i] = data["length_minutes"][mask][0]
    return lengths, rates


lengths_plain, rates_plain = bucket_success_rates(data_plain, n_buckets)
lengths_plus, rates_plus = bucket_success_rates(data_plus, n_buckets)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lengths_plain, rates_plain, "o-", label="plain")
ax.plot(lengths_plus, rates_plus, "s-", label="ReAct++")
ax.axhline(0.5, ls=":", color="gray")
ax.axvline(30.0, ls="--", color="C0", alpha=0.5, label="true horizon plain (30 min)")
ax.axvline(60.0, ls="--", color="C1", alpha=0.5, label="true horizon plus (60 min)")
ax.set_xscale("log")
ax.set_xlabel("task length (minutes, log scale)")
ax.set_ylabel("empirical success rate")
ax.set_title("Per-bucket success rates by scaffold")
ax.legend()
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## 5. Logistic MLE fit

We fit $P(\text{success} \mid L) = \sigma((\mu - \log L)\beta)$ by maximizing the Bernoulli log-likelihood. We use L-BFGS-B with an analytic gradient so the bootstrap inner loop below stays fast. The 50%-horizon point estimate is $\exp(\hat\mu)$.

In [ ]:
def _neg_ll_and_grad(params, log_len, y):
    mu, beta_ = params
    z = (mu - log_len) * beta_
    log_p = -np.logaddexp(0.0, -z)
    log_1mp = -np.logaddexp(0.0, z)
    nll = -(y * log_p + (1.0 - y) * log_1mp).sum()
    p = 1.0 / (1.0 + np.exp(-z))
    r = p - y
    d_mu = (r * beta_).sum()
    d_beta = (r * (mu - log_len)).sum()
    return nll, np.array([d_mu, d_beta])


def fit_logistic_arrays(log_len, y, x0=(3.4, 1.0)):
    res = minimize(
        _neg_ll_and_grad, np.array(x0, dtype=np.float64), args=(log_len, y),
        jac=True, method="L-BFGS-B",
        bounds=[(-5.0, 15.0), (0.05, 10.0)],
        options={"maxiter": 200, "ftol": 1e-9, "gtol": 1e-7},
    )
    return float(res.x[0]), float(res.x[1])


def fit_logistic(data):
    log_len = np.log(data["length_minutes"])
    y = data["success"].astype(np.float64)
    mu_hat, beta_hat = fit_logistic_arrays(log_len, y)
    horizon_hat = float(np.exp(mu_hat))
    return mu_hat, beta_hat, horizon_hat


mu_plain_hat, beta_plain_hat, H_plain_hat = fit_logistic(data_plain)
mu_plus_hat,  beta_plus_hat,  H_plus_hat  = fit_logistic(data_plus)
log_ratio_point = mu_plus_hat - mu_plain_hat

print(f"plain:  mu_hat = {mu_plain_hat:+.4f}   beta_hat = {beta_plain_hat:.4f}   horizon_hat = {H_plain_hat:.2f} min")
print(f"plus:   mu_hat = {mu_plus_hat:+.4f}   beta_hat = {beta_plus_hat:.4f}   horizon_hat = {H_plus_hat:.2f} min")
print(f"point-estimate log-ratio: {log_ratio_point:.4f}")
print(f"  compare true log-ratio: {mu_plus - mu_plain:.4f}")

## 6. Hierarchical bootstrap

Three nested resamples per replicate: buckets, then tasks within bucket, then attempts within task. We reuse the same sampled indices for both scaffolds (paired bootstrap) so task-level jitter cancels the way a real paired design would. `B = 2000` replicates.

Implementation note: we precompute a `(n_buckets, tasks_per_bucket, attempts_per_task)` index tensor for each scaffold, then drive the whole resample by vectorized fancy-indexing instead of triple-nested Python loops. That turns the per-replicate cost into basically "one fit per scaffold".

In [ ]:
def build_cube(data, n_buckets, tasks_per_bucket, attempts_per_task):
    """Return (log_len_cube, y_cube) each with shape (n_buckets, tasks_per_bucket, attempts_per_task)."""
    shp = (n_buckets, tasks_per_bucket, attempts_per_task)
    log_len_cube = np.log(data["length_minutes"]).reshape(shp)
    y_cube = data["success"].astype(np.float64).reshape(shp)
    return log_len_cube, y_cube


def hierarchical_bootstrap_log_ratio(data_a, data_b, n_buckets, tasks_per_bucket, attempts_per_task, B, rng):
    log_len_a, y_a = build_cube(data_a, n_buckets, tasks_per_bucket, attempts_per_task)
    log_len_b, y_b = build_cube(data_b, n_buckets, tasks_per_bucket, attempts_per_task)

    bi = np.arange(n_buckets)[:, None, None]
    tj = np.arange(tasks_per_bucket)[None, :, None]
    # (we only need the bi index; task/attempt indices come from the resample)

    log_ratios = np.empty(B)
    for rep in range(B):
        boot_buckets = rng.integers(0, n_buckets, size=n_buckets)                           # (B_b,)
        boot_tasks = rng.integers(0, tasks_per_bucket, size=(n_buckets, tasks_per_bucket))  # (B_b, T)
        boot_attempts = rng.integers(0, attempts_per_task,
                                      size=(n_buckets, tasks_per_bucket, attempts_per_task))

        # Fancy-indexing: for each (new_i, new_j, new_k) cell we read from
        # cube[boot_buckets[new_i], boot_tasks[new_i, new_j], boot_attempts[new_i, new_j, new_k]].
        b_idx = boot_buckets[:, None, None]                          # (B_b, 1, 1)
        # task index: need boot_tasks[new_i, new_j] but indexed into the *original* bucket boot_buckets[new_i]
        t_idx = boot_tasks                                            # (B_b, T)
        t_idx_e = t_idx[:, :, None]                                   # (B_b, T, 1)
        a_idx = boot_attempts                                         # (B_b, T, A)

        ll_a = log_len_a[b_idx, t_idx_e, a_idx].ravel()
        yy_a = y_a[b_idx, t_idx_e, a_idx].ravel()
        ll_b = log_len_b[b_idx, t_idx_e, a_idx].ravel()
        yy_b = y_b[b_idx, t_idx_e, a_idx].ravel()

        mu_a, _ = fit_logistic_arrays(ll_a, yy_a)
        mu_b, _ = fit_logistic_arrays(ll_b, yy_b)
        log_ratios[rep] = mu_b - mu_a
    return log_ratios

## 7. Run the bootstrap and compute the CI

In [ ]:
rng_boot = np.random.default_rng(seed)
log_ratios = hierarchical_bootstrap_log_ratio(
    data_plain, data_plus,
    n_buckets=n_buckets,
    tasks_per_bucket=tasks_per_bucket,
    attempts_per_task=attempts_per_task,
    B=B_bootstrap,
    rng=rng_boot,
)

ci_lo, ci_hi = np.percentile(log_ratios, [2.5, 97.5])
one_sided_lo = np.percentile(log_ratios, 100 * alpha)
reject_null = one_sided_lo > 0.0
reject_practical = one_sided_lo > practical_threshold_log_ratio

print(f"Point-estimate log-ratio:   {log_ratio_point:.4f}")
print(f"95% percentile CI:          [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"One-sided lower bound (alpha={alpha}): {one_sided_lo:.4f}")
print(f"Reject H0 (lower bound > 0):              {reject_null}")
print(f"Meets practical threshold (> log(1.5)):   {reject_practical}")
print()
print(f"In horizon units:")
print(f"  plain horizon:   {H_plain_hat:.2f} min")
print(f"  plus  horizon:   {H_plus_hat:.2f} min")
print(f"  horizon ratio:   {np.exp(log_ratio_point):.3f}x  (95% CI [{np.exp(ci_lo):.3f}x, {np.exp(ci_hi):.3f}x])")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(log_ratios, bins=40, alpha=0.7, color="C2")
ax.axvline(0.0, ls="--", color="red", label="H0: log-ratio = 0")
ax.axvline(practical_threshold_log_ratio, ls="--", color="orange", label="log(1.5) practical threshold")
ax.axvline(np.log(2.0), ls="--", color="gray", label="log(2) true effect")
ax.axvline(log_ratio_point, ls="-", color="black", label=f"point estimate = {log_ratio_point:.3f}")
ax.set_xlabel("bootstrap log-ratio of horizons")
ax.set_ylabel("count")
ax.set_title(f"Hierarchical bootstrap distribution  (B = {B_bootstrap})")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## 8. Decision

Using the pre-registered rule: check whether the one-sided lower bound clears 0 (statistical significance) and $\log(1.5)$ (practical significance). Because the true effect is $\log(2) \approx 0.693$ and the practical threshold is $\log(1.5) \approx 0.405$, the gap is only about 0.29 nats; whether the lower bound clears the practical threshold is genuinely a design question about whether `tasks_per_bucket = 20` is enough.

How the decision would change:
- If the lower bound straddled 0 (e.g. `[-0.10, 0.50]`), we would **fail to reject** H0 and report the result as inconclusive. The right next step would be more tasks per bucket (see sample-size sensitivity below), not tweaking the analysis.
- If the lower bound were above 0 but below $\log(1.5)$, we would call ReAct++ **statistically but not practically significant** and *not* ship it. This is the failure mode pre-registration is explicitly designed to prevent: without a pre-committed threshold it is very tempting to retroactively re-define "meaningful" to match whatever CI we got.
- If the lower bound cleared both 0 and $\log(1.5)$, we ship.

## 9. Sample-size sensitivity: Monte-Carlo power curve

For each candidate `tasks_per_bucket`, repeat the full experiment many times and count how often the bootstrap lower bound exceeds $\log(1.5)$. We use `B = 400` inside the bootstrap and a modest number of outer reps to keep runtime reasonable. Because the effect-to-threshold gap is only 0.29 nats, we explore a wider `tpb_grid` up through 96 to actually see power cross 80%.

In [ ]:
def run_single_experiment(tpb, B_inner, rng_outer):
    eps = rng_outer.normal(0.0, 0.15, size=(n_buckets, tpb))
    d_plain = generate_eval_data(mu_plain, beta, length_buckets_minutes, tpb, attempts_per_task, eps, rng_outer)
    d_plus  = generate_eval_data(mu_plus,  beta, length_buckets_minutes, tpb, attempts_per_task, eps, rng_outer)

    lr = hierarchical_bootstrap_log_ratio(
        d_plain, d_plus, n_buckets, tpb, attempts_per_task, B=B_inner, rng=rng_outer
    )
    one_sided = np.percentile(lr, 100 * alpha)
    return one_sided > practical_threshold_log_ratio


tpb_grid = [8, 16, 24, 32, 48, 64, 96]
outer_reps = 30
B_inner = 300

power_est = np.zeros(len(tpb_grid))
rng_master = np.random.default_rng(seed + 99)
for i, tpb in enumerate(tpb_grid):
    wins = 0
    for _ in range(outer_reps):
        wins += int(run_single_experiment(tpb, B_inner, rng_master))
    power_est[i] = wins / outer_reps
    print(f"tasks_per_bucket = {tpb:3d}   power ~ {power_est[i]:.3f}")

above = np.where(power_est >= power_target)[0]
min_tpb_for_power = int(tpb_grid[above[0]]) if len(above) else None

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(tpb_grid, power_est, "o-", label="estimated power")
ax.axhline(power_target, ls="--", color="gray", label=f"target {power_target:.0%}")
if min_tpb_for_power is not None:
    ax.axvline(min_tpb_for_power, ls=":", color="red", label=f"min tpb = {min_tpb_for_power}")
ax.set_xlabel("tasks_per_bucket")
ax.set_ylabel("power vs log(1.5)")
ax.set_title(f"Monte-Carlo power (outer reps = {outer_reps}, inner B = {B_inner})")
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nMinimum tasks_per_bucket achieving >= {power_target:.0%} power: {min_tpb_for_power}")

## 10. Sensitivity: coarser bucket scheme

Re-run the experiment with only four log-spaced buckets. If the conclusion flips, our original result was an artifact of binning choice; if it survives, the design is robust.

In [ ]:
coarse_buckets = [1, 10, 60, 600]
n_buckets_coarse = len(coarse_buckets)

rng_coarse_eps = np.random.default_rng(seed + 500)
coarse_eps = rng_coarse_eps.normal(0.0, 0.15, size=(n_buckets_coarse, tasks_per_bucket))
rng_cp = np.random.default_rng(seed + 501)
rng_cq = np.random.default_rng(seed + 502)
coarse_plain = generate_eval_data(mu_plain, beta, coarse_buckets, tasks_per_bucket, attempts_per_task, coarse_eps, rng_cp)
coarse_plus  = generate_eval_data(mu_plus,  beta, coarse_buckets, tasks_per_bucket, attempts_per_task, coarse_eps, rng_cq)

mu_cp, beta_cp, H_cp = fit_logistic(coarse_plain)
mu_cq, beta_cq, H_cq = fit_logistic(coarse_plus)
coarse_log_ratio = mu_cq - mu_cp

print("Coarse (4-bucket) fit:")
print(f"  plain horizon: {H_cp:.2f} min  (fine: {H_plain_hat:.2f})")
print(f"  plus  horizon: {H_cq:.2f} min  (fine: {H_plus_hat:.2f})")
print(f"  log-ratio:     {coarse_log_ratio:.4f}   (fine: {log_ratio_point:.4f})")
print(f"  horizon ratio: {np.exp(coarse_log_ratio):.3f}x  (fine: {np.exp(log_ratio_point):.3f}x)")
print()
print("Conclusion: the point estimate moves a little with binning choice, but the qualitative")
print("answer (~2x horizon improvement) is robust to the binning granularity.")

## 11. Flashcard: 90-second pitch

> *How did you design this eval?*
>
> I was comparing a new agent scaffold, ReAct++, against a plain baseline on an agentic-coding suite using METR's 50%-horizon framing. Before generating any data I pre-registered a one-sided $\alpha = 0.05$ test of $\log(H_{\text{plus}} / H_{\text{plain}}) > 0$, a practical threshold of $\log(1.5) \approx 0.405$, and a target of 80% power against a 2x effect ($\log(2) \approx 0.693$).
>
> The suite had 10 log-spaced length buckets from 1 minute to 16 hours, 20 tasks per bucket, and 3 attempts per task, paired across the two scaffolds on the same per-task difficulty draws. I fit a two-parameter logistic per scaffold by MLE with L-BFGS-B and an analytic gradient, and read off each horizon as $\exp(\hat\mu)$. The fitted horizons came out to about **29 minutes** for plain and **53 minutes** for ReAct++, a point-estimate log-ratio of about **0.60** (true value 0.69).
>
> For uncertainty I ran a three-level hierarchical bootstrap (buckets -> tasks -> attempts) with $B = 2000$, paired across scaffolds on the same resampled indices so task-level jitter cancels. The 95% percentile CI on the log-ratio came out to about **[0.13, 1.09]**, with a one-sided lower bound at $\alpha = 0.05$ of about **0.20**. That clears 0 comfortably but *misses* $\log(1.5) \approx 0.41$, which is the honest answer: at n = 20 tasks per bucket this design has enough power to declare statistical significance against the null, but not enough to declare practical significance against a 50% horizon improvement.
>
> The Monte-Carlo power curve confirmed it: we need roughly **64 tasks per bucket** to cross 80% power against the practical threshold. And a sensitivity re-fit with only 4 coarse buckets gave a horizon ratio of about 1.89x, essentially the same as the 1.82x from the 10-bucket fine grid — so the design is robust to binning choice but *not* robust to the sample size we pre-committed to. Takeaway: the 10x20x3 design is fine for the null test but under-powered for the practical-threshold test, and we should have pre-registered ~60 tasks per bucket instead.

## 12. Interview rehearsal

**Q: "Walk me through how you'd design an eval to compare two agent scaffolds."**

> I'd anchor the design on a single, well-defined estimand. In this case the estimand is the log-ratio of 50%-success time horizons between the two scaffolds, so the entire analysis pipeline has to produce an honest CI on that one number. I pre-register: one-sided $\alpha = 0.05$, power target 80% against a 2x effect, and a practical significance threshold of $\log(1.5)$ so we don't ship a scaffold whose improvement rounds to nothing. I also pre-commit to the full task set before touching any data so there's no peek-and-stop.
>
> For the suite I use 10 log-spaced length buckets from 1 minute to 16 hours, 20 tasks per bucket, 3 attempts per task, and I pair the two scaffolds on the same task IDs so task-level difficulty cancels. Bernoulli per attempt, with a small per-task Gaussian difficulty jitter so the middle resampling level actually has variance.
>
> For the fit I do MLE of a two-parameter logistic in $\mu$ and slope $\beta$, per scaffold, using L-BFGS-B with an analytic gradient so the bootstrap stays cheap. In the notebook I got roughly **29 minutes** for plain and **53 minutes** for ReAct++, a point-estimate log-ratio of about **0.60** against a ground truth of $\log(2) \approx 0.69$.
>
> For uncertainty I use a three-level hierarchical bootstrap: resample bucket indices, then task indices within each sampled bucket, then attempt indices within each sampled task, paired across scaffolds on the same draws. $B = 2000$, 95% percentile CI. My CI on the log-ratio came out to about **[0.13, 1.09]**, with a one-sided lower bound near **0.20**. That clears 0 but not $\log(1.5) \approx 0.41$, so under the pre-registered rule I'd call the result statistically significant but *not* practically significant — and I would say exactly that to the PM.
>
> Two things I always add. One: a Monte-Carlo power curve over `tasks_per_bucket` so I can tell the PM exactly how many more tasks we'd need. In this notebook the curve crosses 80% power against the practical threshold at about **64 tasks per bucket**, roughly 3x what we ran. Two: a sensitivity check on binning granularity; I re-fit with a coarse 4-bucket grid and got a horizon ratio of 1.89x versus the fine-grid 1.82x, which tells me the result isn't an artifact of how I chopped up the length axis.
>
> The honest takeaway from this specific run is that the 10x20x3 design is under-powered for the *practical* test even though it looks fine for the *null* test. Pre-registration is what lets me say that cleanly instead of redefining the threshold to match what I got.